# Foubert AI — Exploratory Data Analysis
**Dataset:** 3 days — 30 april, 1 mei (feestdag), 2 mei 2026  
**Goal:** Identify key signals for predicting where/when to sell ice cream and how much to bring.

## 0 — Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import glob

plt.rcParams.update({
    'figure.facecolor':   'white',
    'axes.facecolor':     '#F8F7F4',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.spines.left':   False,
    'axes.spines.bottom': False,
    'axes.grid':          True,
    'grid.color':         'white',
    'grid.linewidth':     1.2,
    'font.family':        'sans-serif',
    'font.size':          11,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.labelsize':     11,
    'xtick.bottom':       False,
    'ytick.left':         False,
})

BLUE  = '#3266AD'
GRAY  = '#73726C'
AMBER = '#D07A2A'
GREEN = '#1D9E75'
TEAL  = '#5DCAA5'
RED   = '#E24B4A'

DAY_COLORS = [GRAY, RED, BLUE]
DAYS       = ['30 apr\n(weekday)', '1 mei\n(holiday)', '2 mei\n(saturday)']
DATES      = ['2026-04-30', '2026-05-01', '2026-05-02']
DAY_TYPES  = ['weekday', 'holiday', 'weekend']

print('Libraries loaded.')

## 1 — Load all data
Set `BASE` to the root folder of the export (the folder that contains `2026-04-30/`, `2026-05-01/`, `2026-05-02/`).

In [ ]:
BASE = 'foubertai_export'   # ← adjust if your folder is named differently

def load(filename, **kwargs):
    """Load a TSV file for all 3 days and tag each row with date + day_type."""
    frames = []
    for date, day_type in zip(DATES, DAY_TYPES):
        df = pd.read_csv(f'{BASE}/{date}/{filename}', sep='\t', **kwargs)
        df['date']     = date
        df['day_type'] = day_type
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

ts = {'parse_dates': ['datetime_start', 'datetime_stop']}

sales        = load('02_sales.tsv',        **ts)
sale_orders  = load('03_sale_orders.tsv')
shifts       = load('01_shifts.tsv')
calls        = load('07_calls.tsv')
reservations = load('06_reservations.tsv')
vans         = load('08_vans.tsv')

sales['hour'] = sales['datetime_start'].dt.hour

# GPS — all van files across all 3 days
gps_frames = []
for date, day_type in zip(DATES, DAY_TYPES):
    for f in glob.glob(f'{BASE}/{date}/gps/van_*.tsv'):
        df = pd.read_csv(f, sep='\t', parse_dates=['created_at'])
        df['date']     = date
        df['day_type'] = day_type
        gps_frames.append(df)
gps = pd.concat(gps_frames, ignore_index=True)

print(f'sales:         {len(sales):,} rows')
print(f'sale_orders:   {len(sale_orders):,} rows')
print(f'calls:         {len(calls):,} rows')
print(f'reservations:  {len(reservations):,} rows')
print(f'gps points:    {len(gps):,} rows')

## 2 — Build summary table

In [ ]:
items_per_day = (
    sale_orders
    .merge(sales[['sale_id', 'date']], on='sale_id', how='left')
    .groupby('date')
    .size()
    .rename('sale_items')
)

vans_per_day = shifts.groupby('date')['icecream_van_id'].nunique().rename('vans')

summary = (
    sales.groupby('date').size().rename('sales').to_frame()
    .join(calls.groupby('date').size().rename('calls'))
    .join(reservations.groupby('date').size().rename('reservations'))
    .join(items_per_day)
    .join(vans_per_day)
    .join(gps.groupby('date').size().rename('gps_points'))
)
summary['basket_size']   = (summary['sale_items'] / summary['sales']).round(2)
summary['calls_per_van'] = (summary['calls']      / summary['vans']).round(1)
summary['day_type']      = DAY_TYPES

summary

## 3 — KPI overview

In [ ]:
calls['was_assigned'] = calls['shift_id'].notna()
unmet_pct  = (~calls['was_assigned']).mean() * 100
unmet_n    = (~calls['was_assigned']).sum()
peak_day   = summary['sales'].idxmax()

kpis = [
    ('Total sales (3 days)',   f"{summary['sales'].sum():,}",         ''),
    ('Peak day sales',         f"{summary['sales'].max():,}",         peak_day),
    ('Total calls',            f"{summary['calls'].sum():,}",         ''),
    ('Calls unmet (all days)', f"{unmet_pct:.0f}%",                   f"{unmet_n} calls"),
    ('Total reservations',     f"{summary['reservations'].sum():,}",  ''),
    ('Max basket size',        f"{summary['basket_size'].max():.2f}", summary['basket_size'].idxmax()),
]

fig, axes = plt.subplots(1, len(kpis), figsize=(16, 2.2))
for ax, (label, val, sub) in zip(axes, kpis):
    ax.set_axis_off()
    rect = mpatches.FancyBboxPatch(
        (0.05, 0.05), 0.9, 0.9,
        boxstyle='round,pad=0.02', linewidth=0,
        facecolor='#F0EEE8', transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    ax.text(0.5, 0.72, str(val), ha='center', va='center',
            fontsize=20, fontweight='bold', color='#1a1a1a', transform=ax.transAxes)
    ax.text(0.5, 0.38, label, ha='center', va='center',
            fontsize=9.5, color='#666', transform=ax.transAxes)
    if sub:
        ax.text(0.5, 0.16, sub, ha='center', va='center',
                fontsize=8.5, color='#999', transform=ax.transAxes)

fig.suptitle('3-day KPI overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4 — Sales, calls & reservations per day (absolute + indexed)

In [ ]:
x, w = np.arange(3), 0.25
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, indexed, title in zip(
    axes, [False, True],
    ['Absolute volumes per day', 'Indexed (weekday = 100)']):

    s = summary['sales'].values.astype(float)
    c = summary['calls'].values.astype(float)
    r = summary['reservations'].values.astype(float)

    if indexed:
        s = (s / s[0] * 100).round(1)
        c = (c / c[0] * 100).round(1)
        r = (r / r[0] * 100).round(1)

    b1 = ax.bar(x - w, s, w, label='Sales',       color=BLUE,  zorder=3)
    b2 = ax.bar(x,     c, w, label='Calls',        color=GRAY,  zorder=3)
    b3 = ax.bar(x + w, r, w, label='Reservations', color=AMBER, zorder=3)

    for bars in [b1, b2, b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2,
                    h + (8 if not indexed else 1.5),
                    f'{h:.0f}', ha='center', va='bottom', fontsize=8.5, color='#444')

    ax.set_xticks(x)
    ax.set_xticklabels(DAYS)
    ax.set_title(title)
    if indexed:
        ax.set_ylabel('Index (weekday = 100)')
        ax.axhline(100, color='#bbb', linewidth=1, linestyle='--', zorder=2)
    else:
        ax.set_ylabel('Count')
    ax.legend(frameon=False)

plt.suptitle('Sales vs calls vs reservations per day', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 — Demand vs supply gap (unanswered calls)

In [ ]:
call_gap = (
    calls.groupby('date')['was_assigned']
    .agg(served=lambda x: x.sum(),
         no_van=lambda x: (~x).sum(),
         total='count')
    .reset_index()
)
call_gap['unmet_pct'] = (call_gap['no_van'] / call_gap['total'] * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Stacked bar: served vs unmet per day
ax1.bar(DAYS, call_gap['served'], color=BLUE, zorder=3, label='Served', width=0.5)
ax1.bar(DAYS, call_gap['no_van'], color=RED,  zorder=3, label='No van', width=0.5,
        bottom=call_gap['served'])
for i, row in call_gap.iterrows():
    ax1.text(i, row['total'] + 5, f"{row['unmet_pct']:.0f}% unmet",
             ha='center', fontsize=9, color='#444')
ax1.set_ylabel('Number of calls')
ax1.set_title('Calls served vs unmet per day', fontweight='bold')
ax1.legend(frameon=False)

# Calls per active van
cpv = summary['calls_per_van'].values
bars = ax2.bar(DAYS, cpv, color=DAY_COLORS, zorder=3, width=0.5)
for bar, v in zip(bars, cpv):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{v:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.axhline(cpv[0], color='#aaa', linewidth=1, linestyle='--', label='Weekday baseline')
ax2.set_ylabel('Calls per active van')
ax2.set_title('Calls per van per day', fontweight='bold')
ax2.legend(frameon=False)

plt.suptitle('Demand vs supply gap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 — Basket size (items per sale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x, w = np.arange(3), 0.35
b1 = ax.bar(x - w/2, summary['sales'],      w, label='Sales (tickets)', color=BLUE, zorder=3)
b2 = ax.bar(x + w/2, summary['sale_items'], w, label='Items sold',       color=TEAL, zorder=3)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 30, f'{h:,.0f}',
                ha='center', va='bottom', fontsize=8.5, color='#444')
ax.set_xticks(x)
ax.set_xticklabels(DAYS)
ax.set_ylabel('Count')
ax.set_title('Sales tickets vs items sold', fontweight='bold')
ax.legend(frameon=False)

ax2 = axes[1]
bs = summary['basket_size'].values
ax2.plot(DAYS, bs, marker='o', markersize=9, linewidth=2.5, color=BLUE, zorder=3)
ax2.fill_between(DAYS, bs, bs.min() - 0.2, alpha=0.08, color=BLUE)
for d, v in zip(DAYS, bs):
    ax2.annotate(f'{v:.2f}', (d, v), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=11, fontweight='bold', color=BLUE)
ax2.set_ylim(bs.min() - 0.5, bs.max() + 0.5)
ax2.set_ylabel('Items per sale')
ax2.set_title('Avg basket size per day', fontweight='bold')
ax2.axhline(bs[0], color='#aaa', linewidth=1, linestyle='--', label='Weekday baseline')
ax2.legend(frameon=False)

plt.suptitle('Basket size analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Saturday basket is {(bs[2]/bs[0]-1)*100:.0f}% larger than the weekday baseline.')

## 7 — Reservations per day by type

In [ ]:
res_types = (
    reservations
    .groupby(['date', 'reservation_type'])
    .size()
    .unstack(fill_value=0)
)

colors_res = [BLUE, AMBER, GREEN, TEAL, GRAY]
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(3)

for col, color in zip(res_types.columns, colors_res):
    vals = res_types[col].values
    bars = ax.bar(DAYS, vals, bottom=bottom,
                  label=col.replace('_', ' ').title(),
                  color=color, zorder=3, width=0.5)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    str(v), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    bottom += vals

ax.set_ylabel('Number of reservations')
ax.set_title('Reservations per day by type', fontweight='bold')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.show()

## 8 — Hourly sales pattern

In [ ]:
hourly = (
    sales
    .groupby(['date', 'hour'])
    .size()
    .unstack(level=0)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(13, 5))
for col, color, label in zip(hourly.columns, DAY_COLORS, DAYS):
    ax.plot(hourly.index, hourly[col], marker='o', markersize=5,
            linewidth=2, color=color, label=label)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Sales per hour')
ax.set_xticks(range(int(hourly.index.min()), int(hourly.index.max()) + 1))
ax.legend(frameon=False)
ax.set_title('Hourly sales pattern per day', fontweight='bold')
plt.tight_layout()
plt.show()

## 9 — Top selling products

In [ ]:
top10 = sale_orders['name'].value_counts().head(10).index.tolist()

top_pivot = (
    sale_orders[sale_orders['name'].isin(top10)]
    .merge(sales[['sale_id', 'date']], on='sale_id', how='left')
    .groupby(['name', 'date'])
    .size()
    .unstack(fill_value=0)
    .loc[top10]
)

fig, ax = plt.subplots(figsize=(12, 6))
x, w = np.arange(len(top10)), 0.25
for i, (col, color, label) in enumerate(zip(top_pivot.columns, DAY_COLORS, DAYS)):
    ax.bar(x + (i - 1) * w, top_pivot[col], w, label=label, color=color, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(top10, rotation=30, ha='right')
ax.set_ylabel('Times ordered')
ax.set_title('Top 10 products by day', fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 10 — Revenue per hour heatmap

In [ ]:
revenue_heatmap = (
    sales
    .groupby(['date', 'hour'])['total_price_vati']
    .sum()
    .unstack(level=0)
    .fillna(0)
)
revenue_heatmap.columns = DAYS

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(revenue_heatmap.T, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Revenue (VAT incl.)')
ax.set_yticks(range(3))
ax.set_yticklabels(DAYS)
ax.set_xticks(range(len(revenue_heatmap)))
ax.set_xticklabels(revenue_heatmap.index, rotation=45)
ax.set_xlabel('Hour of day')
ax.set_title('Revenue per hour heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

## 11 — GPS stop clusters (velocity < 2 km/h)

In [ ]:
stops = gps[gps['velocity'] < 2]

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharex=True, sharey=True)
for ax, date, label, color in zip(axes, DATES, DAYS, DAY_COLORS):
    day_stops = stops[stops['date'] == date]
    ax.scatter(day_stops['longitude'], day_stops['latitude'],
               s=2, alpha=0.15, color=color, linewidths=0)
    ax.set_title(f'{label}\n({len(day_stops):,} stop points)', fontweight='bold')
    ax.set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

plt.suptitle('GPS stop clusters per day (velocity < 2 km/h)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12 — Sale locations coloured by revenue

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharex=True, sharey=True)
for ax, date, label in zip(axes, DATES, DAYS):
    day_sales = sales[sales['date'] == date]
    sc = ax.scatter(
        day_sales['longitude_start'],
        day_sales['latitude_start'],
        c=day_sales['total_price_vati'],
        cmap='YlOrRd', s=25, alpha=0.7, linewidths=0
    )
    plt.colorbar(sc, ax=ax, label='Revenue (VAT incl.)')
    ax.set_title(f'{label}\n({len(day_sales):,} sales)', fontweight='bold')
    ax.set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

plt.suptitle('Sale locations coloured by revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()